![](https://gymnasium.farama.org/_images/cliff_walking.gif)

# Reinforcement Learning
### *Cliff Walking Problem*

**Environment (`env`)**: 4 x 12 grid

**Actions**: ↑, ↓, ←, →  {0, 3}

**Rewards**:
* Normal cells → -1
* Target cell → 0
* Cliff → -100

*(**Note**: Whenever agent encounters cliff, it returns to state **S**)*

---

### State Representation
**State**: 4 * 12 = 48 cells `(row, col)`

To represent the state as a single number:
`state = row * 12 + col`

**S** (3,0) = 3*12 + 0 = 36

---

### Grid Layout

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 |
| | | | | | | | | | | | |
| **S** | **C** | **C** | **C** | **C** | **C** | **C** | **C** | **C** | **C** | **C** | **E** |

*(Note: **S** = Start, **C** = Cliff, **E** = End/Target)*

In [16]:
import gymnasium as gym
import numpy as np
import random

# Gymnasium

### `env` object
Below are the core methods associated with the environment object:

1. **create** → `make()`
2. **show** → `render()`
3. **reset** → `reset()`
4. **close** → `close()`
5. **action** → `step()`

In [17]:
# sample env
env = gym.make("CliffWalking-v1")

In [18]:
print(env.observation_space.n) #states
print(env.action_space.n) #actions

starting_state, _ = env.reset()
print(starting_state)

48
4
36


# SARSA

In [19]:
# Parameters

alpha = 0.5
gamma = 0.99
episodes = 500
epsilon = 0.1

In [20]:
# Q-table => store Q-values

Q = np.zeros((48,4))

In [21]:
# Policy - E-greedy: state -> action

def epsilon_greedy(state):
    if random.random() < epsilon:
        return env.action_space.sample() # random action => Explore
    else:
        return np.argmax(Q[state]) # Exploit

In [22]:
for episode in range(episodes):

    render = (episode % 50 == 0)

    if render:
        env = gym.make("CliffWalking-v1", render_mode = "human")
        print(f"rendering env for episode= {episode+1}/{episodes}")
    else:
        env = gym.make("CliffWalking-v1")

# terminate -> agnt reached E
# truncate -> cut short (max time or max length reached)

    done = False # when episode has ended
    state, _ = env.reset()
    action = epsilon_greedy(state)

    total_reward = 0
    episode_len = 0

    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_action = epsilon_greedy(next_state)

        #SARSA update
        Q[state, action] += alpha * (reward + gamma*(Q[next_state, next_action] - Q[state, action]))

        state = next_state
        action = next_action

        total_reward += reward
        episode_len += 1

    print(f"episode= {episode+1}/{episodes}: total reward = {total_reward} & ep length = {episode_len}")
    env.close()

        

rendering env for episode= 1/500
episode= 1/500: total reward = -63 & ep length = 63
episode= 2/500: total reward = -1767 & ep length = 579
episode= 3/500: total reward = -120 & ep length = 120
episode= 4/500: total reward = -73 & ep length = 73
episode= 5/500: total reward = -95 & ep length = 95
episode= 6/500: total reward = -71 & ep length = 71
episode= 7/500: total reward = -185 & ep length = 86
episode= 8/500: total reward = -79 & ep length = 79
episode= 9/500: total reward = -67 & ep length = 67
episode= 10/500: total reward = -41 & ep length = 41
episode= 11/500: total reward = -47 & ep length = 47
episode= 12/500: total reward = -41 & ep length = 41
episode= 13/500: total reward = -88 & ep length = 88
episode= 14/500: total reward = -153 & ep length = 54
episode= 15/500: total reward = -572 & ep length = 176
episode= 16/500: total reward = -148 & ep length = 49
episode= 17/500: total reward = -153 & ep length = 54
episode= 18/500: total reward = -78 & ep length = 78
episode= 19

In [23]:
# what did our agent learn?

env = gym.make("CliffWalking-v1", render_mode="human")
done = False 
state, _ = env.reset()
total_reward = 0
episode_len = 0

while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    total_reward += reward
    episode_len += 1

print(f"total reward = {total_reward} & ep length = {episode_len}")
env.close()

total reward = -17 & ep length = 17


In [24]:
import os
import warnings

# 1. Trick PyGame into using a fake screen and fake speakers
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

# 2. Tell Python to hush the harmless MoviePy syntax warning
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- Now proceed with the normal imports and code ---
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import RecordVideo
from IPython.display import Video

# Create the environment in "rgb_array" mode
base_env = gym.make("CliffWalking-v1", render_mode="rgb_array")

# Wrap the environment to automatically record a video
env = RecordVideo(base_env, video_folder='./video', disable_logger=True)

done = False 
state, _ = env.reset()
total_reward = 0
episode_len = 0

while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    total_reward += reward
    episode_len += 1

print(f"total reward = {total_reward} & ep length = {episode_len}")

env.close()

video_path = './video/rl-video-episode-0.mp4'

if os.path.exists(video_path):
    display(Video(video_path, embed=True))
else:
    print("Video file not found.")

total reward = -17 & ep length = 17


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:296: UserWarning: WARN: Overwriting existing videos at /kaggle/working/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
